[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Proyecto integrador: análisis exploratorio de datos.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · **Plantilla del Proyecto integrador**

**Autor(a):** _(tu nombre)_ · **Dataset:** _(nombre y fuente)_ · **Fecha:** _(fecha de entrega)_

> 📌 **Cómo usar esta plantilla.**
> 1. Cópiala con un nombre propio (por ejemplo, `proyecto_diamonds.ipynb`). Si la copias a la raíz del repositorio, cambia en la primera celda `../img/pythonista.png` por `img/pythonista.png`.
> 2. Coloca tu archivo de datos en `data/` y escribe su nombre en `ARCHIVO_PROYECTO` (sección 2). Mientras no lo tengas, la plantilla usa `mpg.csv` como ejemplo, así que corre completa desde el primer día.
> 3. Reemplaza cada _(texto en cursivas entre paréntesis)_ y cada `# TU CÓDIGO AQUÍ`.
> 4. Antes de entregar: **Restart Kernel and Run All Cells** y exporta el informe con `jupyter nbconvert --to html --no-input <tu_notebook>.ipynb`.

**Entrega (Classroom, Evaluación semanal 2, 70 puntos):** el `.ipynb` ejecutado, el informe (`.html` o `.pdf`) y el enlace a tu repositorio de GitHub. Cierre: **domingo 1 de noviembre de 2026, 11:59 pm (CDMX)**.

| Criterio de la rúbrica | Puntos | Sección de esta plantilla |
|---|---|---|
| Preguntas y contexto | 6 | 1 y 2 |
| Calidad de datos | 10 | 3 |
| Descripción y visualización | 12 | 4 y 5 |
| Inferencia | 16 | 6 |
| Informe de hallazgos | 16 | 7 |
| Reproducibilidad | 10 | Todo el *notebook* |

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, definirá `DATA` y la semilla `rng`, y las funciones reutilizables de la sesión 7 (`perfil_rapido`, `resumen_por_grupo`), de la sesión 6 (`d_cohen`) y de la sesión 8 (`contar_palabras`).

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.proportion import proportion_confint


def perfil_rapido(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_faltantes": df.isna().sum(),
        "pct_faltantes": (df.isna().mean() * 100).round(1),
        "n_unicos": df.nunique(dropna=True),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })


def resumen_por_grupo(df, columna, grupo):
    return df.groupby(grupo, observed=True)[columna].agg(
        n="count", media="mean", mediana="median", desv_est="std",
        iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
    ).round(2)


def d_cohen(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)


def contar_palabras(texto):
    return len(texto.split())

---
## 1. Preguntas y contexto.

**Preguntas de análisis** (2 o 3; cada una debe nombrar las variables y poder responderse con los datos):
1. _(pregunta 1: por ejemplo, ¿cómo se relaciona Y con X?)_
2. _(pregunta 2: por ejemplo, ¿difiere Y entre los grupos de G? ¿Qué confusor podría explicarlo?)_
3. _(opcional)_

**¿Para qué decisión sirve responderlas?** _(una o dos frases)_

---
## 2. Datos.

**Fuente:** _(dónde se obtuvieron, con enlace, y su licencia)_ · **Unidad de observación:** _(qué representa cada fila)_ · **Población que representan:** _(a quién se pueden generalizar las conclusiones)_ · **Cómo se recolectaron y sesgos posibles:** _(selección, supervivencia, no respuesta, medición; sesión 1)_

* La siguiente celda leerá el archivo del proyecto desde `data/`; si todavía no existe, usará `mpg.csv` como ejemplo y lo avisará.

In [ ]:
ARCHIVO_PROYECTO = "mi_dataset.csv"  # TU CÓDIGO AQUÍ: nombre de tu archivo en data/
ruta_proyecto = os.path.join(DATA, ARCHIVO_PROYECTO)
usando_ejemplo = not os.path.isfile(ruta_proyecto)
if usando_ejemplo:
    ruta_proyecto = os.path.join(DATA, "mpg.csv")
    print(f"Aviso: no se encontró {ARCHIVO_PROYECTO} en data/; se usa mpg.csv como ejemplo.")
datos_crudos = pd.read_csv(ruta_proyecto)
datos_crudos.shape

* La siguiente celda mostrará el perfil rápido de cada columna.

In [ ]:
perfil_rapido(datos_crudos)

**Diccionario de datos** (sesión 1): una fila por columna que uses.

| Columna | Descripción | Tipo estadístico y escala | Unidad | `dtype` |
|---|---|---|---|---|
| _(columna)_ | _(qué mide)_ | _(nominal, ordinal, discreta, continua; intervalo o razón)_ | _(unidad)_ | _(dtype)_ |

---
## 3. Calidad de datos.

Revisa duplicados (exactos y por llave), faltantes (cuántos, dónde y su posible mecanismo: MCAR, MAR o MNAR), tipos, rangos imposibles, categorías mal escritas y grupos demasiado pequeños (sesión 2). **Nunca modifiques `datos_crudos`:** trabaja sobre `datos`.

* La siguiente celda contará duplicados exactos y por la llave que definas en `LLAVE` (déjala vacía si tu *dataset* no tiene una).

In [ ]:
LLAVE = []  # TU CÓDIGO AQUÍ: por ejemplo ["id"] o ["name", "model_year"]
llave_valida = [c for c in LLAVE if c in datos_crudos.columns]
{"duplicados exactos": int(datos_crudos.duplicated().sum()),
 f"duplicados por {llave_valida}": int(datos_crudos.duplicated(subset=llave_valida).sum()) if llave_valida else None}

* La siguiente celda mostrará el resumen de las columnas numéricas para buscar rangos imposibles.

In [ ]:
datos_crudos.describe().T.round(2)

* La siguiente celda aplicará tus decisiones de limpieza y creará `datos`, sin tocar `datos_crudos`.

In [ ]:
datos = datos_crudos.copy()
# TU CÓDIGO AQUÍ: conversiones de dtype, unificación de categorías, filtros justificados, nuevas variables
filas_antes, filas_despues = len(datos_crudos), len(datos)
{"filas antes": filas_antes, "filas después": filas_despues, "filas eliminadas": filas_antes - filas_despues}

**Decisiones de calidad** (una viñeta por decisión: qué, por qué y cuántas filas o valores afecta):
* _(decisión 1)_
* _(decisión 2)_

**Impacto:** _(¿cambia algún resultado clave si tomas otra decisión razonable? Compáralo al menos en un caso; sesión 2)_

---
## 4. Descripción de distribuciones.

Para cada variable clave: forma (simetría, modas), centro y dispersión **adecuados a la forma** (media y desviación estándar o mediana e IQR) y atípicos (regla de 1.5·IQR o *z* robusto; sesión 3). Define abajo tus variables; con el ejemplo se usan las de `mpg`.

* La siguiente celda definirá la variable numérica principal `Y`, una segunda numérica `X` y una de grupo `G`, eligiendo columnas válidas si las tuyas no existen.

In [ ]:
Y, X, G = "mpg", "weight", "origin"  # TU CÓDIGO AQUÍ: nombres de tus columnas
numericas = datos.select_dtypes("number").columns.tolist()
categoricas = [c for c in datos.columns if c not in numericas and datos[c].nunique() <= 10]
Y = Y if Y in numericas else numericas[0]
X = X if X in numericas and X != Y else next(c for c in numericas if c != Y)
G = G if G in datos.columns else (categoricas[0] if categoricas else None)
Y, X, G

* La siguiente celda dibujará el histograma de `Y` y calculará su asimetría, sus medidas de centro y dispersión y el número de atípicos por la regla de 1.5·IQR.

In [ ]:
valores_y = datos[Y].dropna()
q1, q3 = valores_y.quantile([0.25, 0.75])
iqr = q3 - q1
sns.histplot(valores_y, bins=30).set(xlabel=Y, title=f"Distribución de {Y}")
plt.show()
{"n": len(valores_y), "media": round(float(valores_y.mean()), 2), "mediana": round(float(valores_y.median()), 2),
 "desv_est": round(float(valores_y.std()), 2), "IQR": round(float(iqr), 2),
 "asimetría": round(float(stats.skew(valores_y)), 2),
 "atípicos (1.5·IQR)": int(((valores_y < q1 - 1.5 * iqr) | (valores_y > q3 + 1.5 * iqr)).sum())}

**Interpretación:** _(forma, qué medida de centro reportas y por qué, qué haces con los atípicos)_

---
## 5. Relaciones y comparación de grupos.

Al menos **una comparación de grupos** y **una relación entre variables**, con tablas por grupo (incluyendo el *n*), gráficas rotuladas y la búsqueda de confusores (sesión 4).

* La siguiente celda resumirá `Y` por grupo de `G` (si hay variable de grupo).

In [ ]:
resumen_por_grupo(datos, Y, G) if G else "Define una variable de grupo G en la sección 4."

* La siguiente celda dibujará `Y` contra `X` y calculará las correlaciones de Pearson y de Spearman.

In [ ]:
pares = datos[[X, Y]].dropna()
sns.scatterplot(pares, x=X, y=Y, alpha=0.5).set(title=f"{Y} contra {X}")
plt.show()
{"pearson": round(float(pares[X].corr(pares[Y])), 3), "spearman": round(float(pares[X].corr(pares[Y], method="spearman")), 3)}

**Interpretación:** _(qué se asocia con qué, con qué fuerza y forma, y qué confusor podría explicar la comparación de grupos)_

---
## 6. Inferencia.

Al menos **dos pruebas de hipótesis distintas** (o una prueba y un *bootstrap*), con supuestos revisados, IC y tamaño del efecto (sesiones 5 y 6). Escribe H0 y H1 **antes** de ejecutar.

* **Prueba 1:** H0: _(…)_ · H1: _(…)_
* **Prueba 2:** H0: _(…)_ · H1: _(…)_

* La siguiente celda comparará `Y` entre los dos grupos más frecuentes de `G` con Levene, Welch, Mann-Whitney y la d de Cohen.

In [ ]:
if G:
    grupo_a, grupo_b = datos[G].value_counts().index[:2]
    valores_a = datos.loc[datos[G] == grupo_a, Y].dropna().to_numpy()
    valores_b = datos.loc[datos[G] == grupo_b, Y].dropna().to_numpy()
    welch = stats.ttest_ind(valores_a, valores_b, equal_var=False)
    ic_welch = welch.confidence_interval()
    resultado_dos_grupos = {
        "grupos": (str(grupo_a), str(grupo_b)), "n": (len(valores_a), len(valores_b)),
        "Levene p": round(float(stats.levene(valores_a, valores_b).pvalue), 4),
        "diferencia de medias": round(float(valores_a.mean() - valores_b.mean()), 3),
        "IC 95% (Welch)": (round(float(ic_welch.low), 3), round(float(ic_welch.high), 3)),
        "Welch p": float(welch.pvalue),
        "Mann-Whitney p": float(stats.mannwhitneyu(valores_a, valores_b).pvalue),
        "d de Cohen": round(float(d_cohen(valores_a, valores_b)), 2),
    }
else:
    resultado_dos_grupos = "Define una variable de grupo G en la sección 4."
resultado_dos_grupos

* La siguiente celda calculará un IC *bootstrap* percentil del 95% para la mediana de `Y` (`random_state=401`).

In [ ]:
ic_mediana = stats.bootstrap((datos[Y].dropna().to_numpy(),), np.median, n_resamples=9999,
                             method="percentile", random_state=401).confidence_interval
round(float(datos[Y].median()), 3), (round(float(ic_mediana.low), 3), round(float(ic_mediana.high), 3))

* La siguiente celda es para tu segunda prueba (por ejemplo, ANOVA de Welch con η², chi cuadrada con V de Cramér o prueba *z* de proporciones).

In [ ]:
# TU CÓDIGO AQUÍ

**Interpretación:** _(para cada prueba: supuestos, resultado, IC, tamaño del efecto y qué significa en lenguaje llano)_

---
## 7. Informe de hallazgos.

> Esta sección es la que se exporta como informe (`--no-input`), así que debe poder leerse **sin el código**. Extensión total: 2 a 4 páginas.

### 7.1 Resumen ejecutivo.

_(máximo 150 palabras: qué se analizó y para qué, la respuesta a cada pregunta con su número clave e incertidumbre, y la principal limitación)_

* La siguiente celda verificará que el resumen ejecutivo no pase de 150 palabras (copia aquí el texto de 7.1).

In [ ]:
resumen_ejecutivo = "TU RESUMEN AQUÍ"
contar_palabras(resumen_ejecutivo)

### 7.2 Datos y método.

_(fuente, población, decisiones de calidad con su número de filas, y métodos usados)_

### 7.3 Hallazgos.

**Hallazgo 1:** _(afirmación en una frase)_
* Gráfica final: _(título con el hallazgo, unidades, n; sesión 8)_
* Estimación e IC: _(…)_ · Tamaño del efecto: _(…)_ · Prueba: _(…)_
* Interpretación en lenguaje llano: _(…)_

**Hallazgo 2:** _(…)_

### 7.4 Limitaciones.

* _(sesgos de recolección, confusores no controlados, supuestos dudosos, lo que no se puede concluir)_

### 7.5 Recomendaciones.

* _(qué hacer con los resultados y qué datos harían falta)_

### 7.6 Reproducibilidad.

Repositorio: _(enlace)_ · Ejecutar con **Restart Kernel and Run All Cells** en *Codespaces*; dependencias en `requirements.txt`; semilla 401.

* La siguiente celda mostrará las versiones de las bibliotecas usadas, para la sección de reproducibilidad.

In [ ]:
import matplotlib
import scipy
import seaborn
import statsmodels
{"pandas": pd.__version__, "numpy": np.__version__, "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
 "matplotlib": matplotlib.__version__, "seaborn": seaborn.__version__, "datos de ejemplo": usando_ejemplo}

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>